In [ ]:
pip install opensmile

In [ ]:
pip install faster-whisper

In [ ]:
from faster_whisper import WhisperModel
import opensmile
import pandas as pd

# Step 1: Transcribe with word-level timestamps
audio_path = "common_voice_en_35786963.wav"
model = WhisperModel("base", device="cpu", compute_type="int8")

segments, _ = model.transcribe(audio_path, word_timestamps=True)

# Extract word-level timestamps
word_timestamps = []
for segment in segments:
    for word in segment.words:
        word_timestamps.append({
            "word": word.word,
            "start": word.start,
            "end": word.end
        })

# Step 2: Extract loudness with openSMILE
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
)

features = smile.process_file(audio_path)

if "Loudness_sma3" not in features.columns:
    raise ValueError("pcm_loudness not found. Check your openSMILE feature set.")
loudness = features["Loudness_sma3"]

# Step 3: Compute overall average loudness

smile_func = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

avg_loudness_audio = smile_func.process_file(audio_path)['loudness_sma3_amean'].iloc[0]

# Step 4: Compute per-word loudness & annotate if emphasized
annotated_words = []

for entry in word_timestamps:
    start, end = entry["start"], entry["end"]
    
    # Get word loudness base on word times
    condition_get_loudness = (loudness.index.get_level_values(1).total_seconds() >= start) & (loudness.index.get_level_values(1).total_seconds() <= end)
    segment =  loudness[condition_get_loudness]
    word_loudness = segment.mean() if not segment.empty else 0

    is_emphasized = word_loudness > avg_loudness_audio

    annotated_words.append({
        "word": entry["word"],
        "loudness": word_loudness,
        "average_loundness": avg_loudness_audio,
        "emphasized": is_emphasized,
    })

# Step 5: Output
df = pd.DataFrame(annotated_words)
print(df[["word", "loudness", "average_loundness", "emphasized"]])


          word  loudness  average_loundness  emphasized
0            I  0.358969           0.801848       False
1        never  1.744271           0.801848        True
2       myself  1.806004           0.801848        True
3          saw  1.398586           0.801848        True
4           or  1.411692           0.801848        True
5        heard  1.306675           0.801848        True
6     anything  1.001219           0.801848        True
7           of  1.065782           0.801848        True
8         such  1.251909           0.801848        True
9   practices.  0.759217           0.801848       False
